# 11 — Choose-period policy selection

**Project:** Hotel overbooking decision support  
**Stage:** Translate cancellation estimates into candidate overbooking counts  
**Purpose:** Compare simple and model-based policies using the client's unchanged scoring contract.

The earlier notebooks estimate cancellation probabilities and nightly cancellation uncertainty. This notebook turns those estimates into an integer overbooking count for each hotel-arrival night.

We compare:
1. No overbooking.
2. A global historical cancellation-rate rule.
3. A hotel-specific historical rate rule.
4. A hotel-and-arrival-month lookup rule.
5. A learned model's expected cancellation count.
6. Learned-model uncertainty buffers added to the expected count.

We select among these policies using **Choose only**. The client's walk ceiling is a hard constraint. The contract's walk cost and empty-room cost remain unchanged.

**Important:** The dataset has booking rows, not room inventory or actual walk records. The scoring contract treats each booking row as a room-equivalent for simulation. Results are contract-based historical simulations, not proof of physical occupancy or actual guest walks.

In [1]:
# Run from the repository root, so `from src import ...` and the relative evidence/ paths work
# whether this notebook is opened from the repo folder or from notebooks/pipeline/.
import os
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "contract.py").exists())
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print("Working directory:", REPO_ROOT)

Working directory: C:\Users\Lenovo\Desktop\Build_2_Repo\IIT_GN_W06


## 1. Load the source data and the scoring contract

We use the original data loader, derive arrival date and stay nights, and split the rows using the supplied contract. Do not edit `src/contract.py`.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from src import data, contract

raw = data.read_raw()
bookings = raw.copy()
bookings["arrival_date"] = data.arrival_date(bookings)
bookings["nights"] = (
    bookings["stays_in_weekend_nights"] + bookings["stays_in_week_nights"]
)

train, choose, proof = contract.split_periods(bookings)

print("Train rows:", f"{len(train):,}")
print("Choose rows:", f"{len(choose):,}")
print("Proof rows:", f"{len(proof):,}")
print("Walk cost:", contract.WALK_COST)
print("Walk ceiling per 1,000 arrivals:", contract.WALK_CEILING)

Train rows: 49,157
Choose rows: 29,546
Proof rows: 40,687
Walk cost: 300.0
Walk ceiling per 1,000 arrivals: 10.0


## 2. Check the evaluation periods

Only Train and Choose are used in this notebook. Proof is not scored or inspected for policy selection.

In [3]:
assert train["arrival_date"].max() < pd.Timestamp(contract.TRAIN_END)
assert choose["arrival_date"].min() >= pd.Timestamp(contract.TRAIN_END)
assert choose["arrival_date"].max() < pd.Timestamp(contract.PROOF_START)
assert proof["arrival_date"].min() >= pd.Timestamp(contract.PROOF_START)

print("Train and Choose boundaries verified.")
print("Proof is reserved for the final evaluation.")

Train and Choose boundaries verified.
Proof is reserved for the final evaluation.


## 3. Define the candidate feature set and selected model

The feature set matches Notebooks 05–10. `country` remains held back pending verification of original booking-time availability.

The cell below reads the model and settings selected by Notebook 08 from `evidence/hyperparameter_experiments/selected_settings.csv`.

In [4]:
categorical_columns = [
    "hotel",
    "market_segment",
    "distribution_channel",
    "agent",
    "company"
]

numeric_columns = [
    "is_repeated_guest",
    "previous_cancellations",
    "previous_bookings_not_canceled"
]

feature_columns = categorical_columns + numeric_columns

# Read the configuration Notebook 08 selected (previously a hand-copied placeholder: logistic regression, C=1.0).
import ast

selected_08 = pd.read_csv("evidence/hyperparameter_experiments/selected_settings.csv").iloc[0]
BASE_MODEL_NAME = selected_08["model"]
BASE_MODEL_SETTINGS = ast.literal_eval(selected_08["settings"])
print("Base model from Notebook 08:", BASE_MODEL_NAME, BASE_MODEL_SETTINGS)

def prepare_features(frame):
    X = frame[feature_columns].copy()

    for column in categorical_columns:
        X[column] = X[column].astype("string")

    X["agent"] = X["agent"].fillna("NO_AGENT")
    X["company"] = X["company"].fillna("NO_COMPANY")

    for column in ["hotel", "market_segment", "distribution_channel"]:
        X[column] = X[column].fillna("UNKNOWN_" + column.upper())

    return X

Base model from Notebook 08: logistic_regression {'C': 0.1}


## 4. Fit the selected model using Train only

This notebook fits the base model on all Train rows. It does not re-fit the calibration mapping from Notebook 09: it applies the mapping Notebook 09 saved, but only if Notebook 09's pre-declared rule adopted calibration (lower Choose Brier score **and** log loss). The next cell checks that the base and calibrated probabilities match Notebook 09's saved values exactly. The decision is recorded in `docs/DECISIONS.md`.

In [5]:
def make_preprocessor():
    return ColumnTransformer(
        transformers=[
            (
                "categorical",
                OneHotEncoder(handle_unknown="ignore"),
                categorical_columns
            ),
            (
                "numeric",
                Pipeline(steps=[
                    ("imputer", SimpleImputer(strategy="median")),
                    ("scaler", StandardScaler())
                ]),
                numeric_columns
            )
        ],
        remainder="drop"
    )


if BASE_MODEL_NAME == "logistic_regression":
    estimator = LogisticRegression(
        C=BASE_MODEL_SETTINGS["C"],
        max_iter=1000
    )
elif BASE_MODEL_NAME == "random_forest":
    estimator = RandomForestClassifier(
        n_estimators=BASE_MODEL_SETTINGS["n_estimators"],
        min_samples_leaf=BASE_MODEL_SETTINGS["min_samples_leaf"],
        random_state=42,
        n_jobs=-1
    )
else:
    raise ValueError("Unsupported base model")

model = Pipeline(steps=[
    ("preprocess", make_preprocessor()),
    ("model", estimator)
])

X_train = prepare_features(train)
y_train = train["is_canceled"].astype(int)
X_choose = prepare_features(choose)

model.fit(X_train, y_train)

classes = list(model.named_steps["model"].classes_)
positive_position = classes.index(1)

model_choose_probability = model.predict_proba(
    X_choose
)[:, positive_position]

print("Model fitted on Train; predictions created for Choose.")

Model fitted on Train; predictions created for Choose.


In [6]:
# Apply Notebook 09's calibration decision (rule fixed before Notebook 09 ran; see calibration_decision.json).
import json

calibration_decision = json.loads(Path("evidence/probability_calibration/calibration_decision.json").read_text(encoding="utf-8"))
assert calibration_decision["base_model"] == BASE_MODEL_NAME and calibration_decision["base_settings"] == BASE_MODEL_SETTINGS, \
    "Notebook 09 calibrated a different base model; rerun Notebooks 08 and 09 first."
USE_CALIBRATED = bool(calibration_decision["use_calibrated_probabilities"])
CAL_INTERCEPT = calibration_decision["calibrator_intercept"]
CAL_COEFFICIENT = calibration_decision["calibrator_coefficient"]


def calibrate(probability):
    return 1.0 / (1.0 + np.exp(-(CAL_INTERCEPT + CAL_COEFFICIENT * np.asarray(probability, dtype=float))))


# Consistency check: this notebook's base model and mapping reproduce Notebook 09's saved Choose probabilities.
saved_09 = pd.read_csv("evidence/probability_calibration/choose_calibrated_probabilities.csv", index_col=0)
assert saved_09.index.equals(choose.index)
assert np.allclose(saved_09["uncalibrated"].to_numpy(), model_choose_probability, atol=1e-8)
assert np.allclose(saved_09["calibrated"].to_numpy(), calibrate(model_choose_probability), atol=1e-8)

model_choose_probability_uncalibrated = model_choose_probability
if USE_CALIBRATED:
    model_choose_probability = calibrate(model_choose_probability_uncalibrated)
PROBABILITY_METHOD = "calibrated" if USE_CALIBRATED else "uncalibrated"
print("Probability method for the model policies:", PROBABILITY_METHOD)
print("Matches Notebook 09's saved Choose probabilities: yes")
print(f"Mean Choose probability: uncalibrated {model_choose_probability_uncalibrated.mean():.3f}, "
      f"used {model_choose_probability.mean():.3f}, actual {choose['is_canceled'].mean():.3f}")

Probability method for the model policies: calibrated
Matches Notebook 09's saved Choose probabilities: yes
Mean Choose probability: uncalibrated 0.285, used 0.360, actual 0.369


## 5. Create historical lookup probabilities using Train

We create three transparent rate estimates:
- one global cancellation rate;
- one rate per hotel;
- one rate per hotel and arrival month.

For a hotel-month combination absent from Train, we fall back to that hotel's Train rate. If a hotel itself is absent, we fall back to the global Train rate.

These lookup estimates are historical baselines. They are not machine-learning predictions.

In [7]:
global_rate = train["is_canceled"].mean()
hotel_rates = train.groupby("hotel")["is_canceled"].mean()

train_with_month = train.assign(
    arrival_month=train["arrival_date"].dt.month
)
choose_with_month = choose.assign(
    arrival_month=choose["arrival_date"].dt.month
)

hotel_month_rates = train_with_month.groupby(
    ["hotel", "arrival_month"]
)["is_canceled"].mean()

lookup = choose_with_month[["hotel", "arrival_month"]].copy()
lookup["global_rate"] = global_rate
lookup["hotel_rate"] = lookup["hotel"].map(hotel_rates).fillna(global_rate)

lookup_keys = pd.MultiIndex.from_frame(
    lookup[["hotel", "arrival_month"]]
)
lookup["hotel_month_rate"] = hotel_month_rates.reindex(
    lookup_keys
).to_numpy()

lookup["hotel_month_rate"] = lookup["hotel_month_rate"].fillna(
    lookup["hotel_rate"]
)

lookup.head()

,hotel,arrival_month,global_rate,hotel_rate,hotel_month_rate
6765,Resort Hotel,7,0.357731,0.251511,0.232221
6766,Resort Hotel,7,0.357731,0.251511,0.232221
6767,Resort Hotel,7,0.357731,0.251511,0.232221
6768,Resort Hotel,7,0.357731,0.251511,0.232221
6769,Resort Hotel,7,0.357731,0.251511,0.232221


## 6. Aggregate booking probabilities to hotel-arrival nights

The contract function validates probability values and calculates:
- bookings due;
- actual cancellations/no-shows;
- expected cancellations;
- cancellation-count spread;
- average booked stay value (`ADR × nights`).

The output is one row per hotel and arrival night. We use the same aggregation function for all probability methods so comparisons remain consistent.

In [8]:
choose_scoring = choose[
    ["hotel", "arrival_date", "is_canceled", "adr", "nights"]
].copy()

nightly_model = contract.nightly_table(
    choose_scoring,
    model_choose_probability
)

global_probability = np.full(len(choose), global_rate)
hotel_probability = choose["hotel"].map(hotel_rates).fillna(global_rate).to_numpy()
lookup_probability = lookup["hotel_month_rate"].to_numpy()

nightly_global = contract.nightly_table(
    choose_scoring,
    global_probability
)

nightly_hotel = contract.nightly_table(
    choose_scoring,
    hotel_probability
)

nightly_lookup = contract.nightly_table(
    choose_scoring,
    lookup_probability
)

display(nightly_model.head())

book  cancelled   expected    spread  stay_value
hotel      arrival_date                                                  
City Hotel 2016-07-01     141         68  58.298882  5.621251  319.747589
           2016-07-02      80         33  27.100576  4.136567  397.106875
           2016-07-03      85         21  29.620610  4.308191  478.769529
           2016-07-04      92         30  29.766845  4.220520  440.365543
           2016-07-05      74         14  23.675281  3.907180  405.705811

## 7. Understand the overbooking count

The contract's `oversell` value is an integer number of extra rooms sold for a hotel-arrival night.

For a night:
- if actual cancellations exceed the oversell count, some rooms remain empty;
- if oversell exceeds actual cancellations, the difference is the simulated number of walked guests.

The expected cancellation count is not itself a room-capacity estimate. The contract is a scoring simulation that assumes each booking row represents one room-equivalent.

## 8. Define policy rules

We compare these rules:
- **No overbooking:** always set the count to zero.
- **Global rate:** round up the global Train cancellation rate multiplied by the night's booking count.
- **Hotel rate:** use the corresponding hotel's Train cancellation rate.
- **Hotel-month lookup:** use the Train rate for that hotel and calendar month, with the documented fallback.
- **Model expected:** round up the model's expected cancellation count.
- **Model uncertainty buffer:** round up expected cancellations plus a multiple of the model spread.

The uncertainty multiplier is selected from a small, declared grid using Choose. A larger multiplier generally reduces empty rooms but may increase walks.

In [9]:
def policy_counts(nightly, multiplier=0.0):
    expected = nightly["expected"]
    spread = nightly["spread"]
    return np.ceil(expected + multiplier * spread).astype(int)


policy_predictions = {
    "no_overbooking": pd.Series(
        np.zeros(len(nightly_model), dtype=int),
        index=nightly_model.index
    ),
    "global_rate": pd.Series(
        np.ceil(nightly_global["expected"]).astype(int),
        index=nightly_global.index
    ),
    "hotel_rate": pd.Series(
        np.ceil(nightly_hotel["expected"]).astype(int),
        index=nightly_hotel.index
    ),
    "hotel_month_lookup": pd.Series(
        np.ceil(nightly_lookup["expected"]).astype(int),
        index=nightly_lookup.index
    ),
    "model_expected": pd.Series(
        policy_counts(nightly_model, multiplier=0.0),
        index=nightly_model.index
    )
}

for multiplier in [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]:
    policy_predictions[f"model_buffer_{multiplier}"] = pd.Series(
        policy_counts(nightly_model, multiplier=multiplier),
        index=nightly_model.index
    )

print("Candidate policies:", list(policy_predictions))

Candidate policies: ['no_overbooking', 'global_rate', 'hotel_rate', 'hotel_month_lookup', 'model_expected', 'model_buffer_0.25', 'model_buffer_0.5', 'model_buffer_0.75', 'model_buffer_1.0', 'model_buffer_1.25', 'model_buffer_1.5', 'model_buffer_2.0']


### 8b. Declared extension: the contract's flat rule, and the model with the same safety margin

*Added 2026-11-18, fixed before this notebook was first run.*

`docs/EVALUATION_CONTRACT.md` defines the client's current rule as *"each hotel's cancellation rate, times the night's bookings, less a fixed number of rooms for safety"*, with the margin *"chosen on the choose period under the ceiling, the same way you choose your own"*, and project rule 5 requires every candidate to be compared with it. The declared policies above include only its zero-margin version (`hotel_rate`), and every model buffer above **adds** rooms (`ceil(expected + k × spread)`), so no candidate takes rooms off for safety.

So two families are added, with the same margin grid and the same rounding as the rules above:
- `flat_rule_margin_m` = max(0, ceil(hotel rate × bookings) − m), for m = 0 … 60: the contract's flat rule;
- `model_margin_m` = max(0, ceil(model expected cancellations) − m), for m = 0 … 60: the model given the same treatment.

All declared policies above are kept unchanged. The uncalibrated model policies are scored for reference only and are **not eligible** for selection, because Notebook 09's rule adopted calibration.

In [10]:
MARGINS = range(0, 61)
hotel_rate_rooms = np.ceil(nightly_hotel["expected"]).astype(int)
model_rooms = np.ceil(nightly_model["expected"]).astype(int)
for margin in MARGINS:
    policy_predictions[f"flat_rule_margin_{margin}"] = pd.Series(np.maximum(hotel_rate_rooms - margin, 0), index=nightly_hotel.index)
    policy_predictions[f"model_margin_{margin}"] = pd.Series(np.maximum(model_rooms - margin, 0), index=nightly_model.index)

# Reference only (not eligible): the declared model policies with uncalibrated probabilities.
nightly_model_uncalibrated = contract.nightly_table(choose_scoring, model_choose_probability_uncalibrated)
reference_predictions = {"uncalibrated_model_expected": policy_counts(nightly_model_uncalibrated, multiplier=0.0)}
for multiplier in [0.25, 0.5, 0.75, 1.0, 1.25, 1.5, 2.0]:
    reference_predictions[f"uncalibrated_model_buffer_{multiplier}"] = policy_counts(nightly_model_uncalibrated, multiplier=multiplier)

print("Eligible candidate policies:", len(policy_predictions))
print("Reference-only policies:", len(reference_predictions))

Eligible candidate policies:

 134
Reference-only policies: 8


## 9. Score every policy using the unchanged contract

We use `contract.night_cost()` rather than reimplementing the client's scoring formula. The walk ceiling is applied to the total Choose period.

The lowest-cost policy is not acceptable if it breaches the walk ceiling.

In [11]:
policy_score_rows = []

for policy_name, oversell_counts in policy_predictions.items():
    score = contract.night_cost(
        nightly_model,
        oversell_counts.to_numpy()
    )

    policy_score_rows.append({
        "policy": policy_name,
        **score
    })

policy_scores = pd.DataFrame(policy_score_rows)
policy_scores = policy_scores.sort_values(
    ["ceiling_met", "cost_eur"],
    ascending=[False, True]
).reset_index(drop=True)

display(policy_scores)

,policy,cost_eur,empty_rooms,walked_guests,walks_per_1000,ceiling_met,nights
0,flat_rule_margin_17,2414409.40,5532,169,9.06,True,368
1,flat_rule_margin_18,2476817.51,5721,149,7.99,True,368
2,flat_rule_margin_19,2537394.64,5905,130,6.97,True,368
3,model_margin_18,2576784.14,5934,182,9.76,True,368
4,flat_rule_margin_20,2596170.54,6085,114,6.11,True,368
...,...,...,...,...,...,...,...
129,flat_rule_margin_15,2275648.93,5126,223,11.96,False,368
130,model_margin_15,2343870.86,5289,228,12.22,False,368
131,flat_rule_margin_16,2346400.01,5333,194,10.40,False,368
132,model_margin_16,2424300.51,5511,210,11.26,False,368


## 10. Select the lowest-cost policy that meets the walk ceiling

The client guardrail is at most the specified number of walked guests per 1,000 actual arrivals. We first filter to policies meeting that constraint, then choose the one with the lowest total cost.

If no candidate meets the guardrail, we must report that fact. We must not quietly relax the client's limit.

In [12]:
acceptable_policies = policy_scores[
    policy_scores["ceiling_met"]
].copy()

if acceptable_policies.empty:
    selected_policy = None
    print("No candidate policy meets the client's walk ceiling.")
else:
    selected_policy = acceptable_policies.sort_values(
        "cost_eur"
    ).iloc[0]

    print("Selected policy:", selected_policy["policy"])
    print("Choose-period cost (EUR):", selected_policy["cost_eur"])
    print("Walks per 1,000 arrivals:", selected_policy["walks_per_1000"])
    print("Walk ceiling met:", selected_policy["ceiling_met"])

Selected policy: flat_rule_margin_17
Choose-period cost (EUR): 2414409.4
Walks per 1,000 arrivals: 9.06
Walk ceiling met: True


In [13]:
# Project rule 5: compare with the contract's flat rule, its margin chosen on Choose the same way.
is_flat = policy_scores["policy"].str.startswith("flat_rule_margin_")
acceptable_flat = policy_scores[is_flat & policy_scores["ceiling_met"]]
best_flat = acceptable_flat.sort_values("cost_eur", kind="stable").iloc[0] if not acceptable_flat.empty else None

comparison_columns = ["policy", "cost_eur", "empty_rooms", "walked_guests", "walks_per_1000", "ceiling_met"]
rows = [r for r in [selected_policy, best_flat] if r is not None]
rule5 = pd.DataFrame([r[comparison_columns] for r in rows])
rule5.index = ["selected policy", "best flat rule (contract baseline)"][:len(rows)]
if selected_policy is not None and best_flat is not None:
    rule5["vs flat rule (EUR)"] = rule5["cost_eur"] - best_flat["cost_eur"]
    rule5["vs flat rule (%)"] = rule5["vs flat rule (EUR)"] / best_flat["cost_eur"]
display(rule5)

reference_scores = pd.DataFrame([{"policy": name, **contract.night_cost(nightly_model_uncalibrated, counts.to_numpy())}
                                 for name, counts in reference_predictions.items()])
print("Reference only, not eligible (uncalibrated model probabilities):")
display(reference_scores)

best_by_family = (policy_scores.assign(family=policy_scores["policy"].str.replace(r"_[0-9.]+$", "", regex=True))
                  .query("ceiling_met").sort_values("cost_eur", kind="stable").groupby("family", sort=False).head(1))
print("Cheapest policy meeting the ceiling, per family:")
display(best_by_family[["family", *comparison_columns]].reset_index(drop=True))

,policy,cost_eur,empty_rooms,walked_guests,walks_per_1000,ceiling_met,vs flat rule (EUR),vs flat rule (%)
selected policy,flat_rule_margin_17,2414409.4,5532,169,9.06,True,0.0,0.0
best flat rule (contract baseline),flat_rule_margin_17,2414409.4,5532,169,9.06,True,0.0,0.0


Reference only, not eligible (uncalibrated model probabilities):


,policy,cost_eur,empty_rooms,walked_guests,walks_per_1000,ceiling_met,nights
0,uncalibrated_model_expected,1319331.64,2874,601,32.22,False,368
1,uncalibrated_model_buffer_0.25,1240549.22,2636,683,36.62,False,368
2,uncalibrated_model_buffer_0.5,1184335.76,2417,807,43.27,False,368
3,uncalibrated_model_buffer_0.75,1123038.25,2197,916,49.11,False,368
4,uncalibrated_model_buffer_1.0,1076040.21,1998,1050,56.29,False,368
5,uncalibrated_model_buffer_1.25,1044900.21,1823,1196,64.12,False,368
6,uncalibrated_model_buffer_1.5,1008489.04,1634,1351,72.43,False,368
7,uncalibrated_model_buffer_2.0,973678.10,1314,1693,90.77,False,368


Cheapest policy meeting the ceiling, per family:


,family,policy,cost_eur,empty_rooms,walked_guests,walks_per_1000,ceiling_met
0,flat_rule_margin,flat_rule_margin_17,2414409.40,5532,169,9.06,True
1,model_margin,model_margin_18,2576784.14,5934,182,9.76,True
2,no_overbooking,no_overbooking,4155340.86,10894,0,0.00,True


## 11. Report policy performance separately by hotel

The overall guardrail is the contract criterion. We also calculate each hotel's cost and walk rate so that a good group-level average does not hide a property-specific problem.

These per-hotel values are diagnostic. Do not replace the client's group-level contract with a different constraint unless the client explicitly asks for that.

In [14]:
hotel_policy_rows = []

for policy_name, oversell_counts in policy_predictions.items():
    for hotel_name in nightly_model.index.get_level_values("hotel").unique():
        hotel_mask = nightly_model.index.get_level_values("hotel") == hotel_name
        hotel_nights = nightly_model.loc[hotel_mask]
        hotel_counts = oversell_counts.loc[hotel_mask]

        score = contract.night_cost(
            hotel_nights,
            hotel_counts.to_numpy()
        )

        hotel_policy_rows.append({
            "policy": policy_name,
            "hotel": hotel_name,
            **score
        })

hotel_policy_scores = pd.DataFrame(hotel_policy_rows)
display(
    hotel_policy_scores.sort_values(
        ["hotel", "ceiling_met", "cost_eur"],
        ascending=[True, False, True]
    )
)

,policy,hotel,cost_eur,empty_rooms,walked_guests,walks_per_1000,ceiling_met,nights
104,flat_rule_margin_20,City Hotel,1173611.17,3505,114,9.53,True,184
108,flat_rule_margin_21,City Hotel,1224575.37,3665,101,8.44,True,184
112,flat_rule_margin_22,City Hotel,1277521.43,3828,91,7.61,True,184
116,flat_rule_margin_23,City Hotel,1330480.77,3990,82,6.85,True,184
106,model_margin_20,City Hotel,1351382.75,3972,113,9.45,True,184
...,...,...,...,...,...,...,...,...
71,model_margin_11,Resort Hotel,1070432.39,1766,104,15.55,False,184
75,model_margin_12,Resort Hotel,1123110.61,1870,91,13.60,False,184
79,model_margin_13,Resort Hotel,1174168.03,1966,83,12.41,False,184
83,model_margin_14,Resort Hotel,1219738.34,2054,76,11.36,False,184


## 12. Inspect the selected policy's nightly decisions

This table shows the selected policy's integer oversell count alongside expected cancellations, uncertainty spread and actual cancellations.

The actual cancellation count is visible only because this is historical Choose-period evaluation. It is not available when making a future operational decision.

In [15]:
if selected_policy is not None:
    selected_name = selected_policy["policy"]
    selected_counts = policy_predictions[selected_name]

    selected_nights = nightly_model.copy()
    selected_nights["oversell_count"] = selected_counts
    selected_nights["empty_rooms_simulated"] = (
        selected_nights["cancelled"] - selected_nights["oversell_count"]
    ).clip(lower=0)
    selected_nights["walks_simulated"] = (
        selected_nights["oversell_count"] - selected_nights["cancelled"]
    ).clip(lower=0)

    display(selected_nights.head(20))
else:
    print("No selected policy because none met the walk ceiling.")

book  cancelled   expected    spread  stay_value  \
hotel      arrival_date                                                     
City Hotel 2016-07-01     141         68  58.298882  5.621251  319.747589   
           2016-07-02      80         33  27.100576  4.136567  397.106875   
           2016-07-03      85         21  29.620610  4.308191  478.769529   
           2016-07-04      92         30  29.766845  4.220520  440.365543   
           2016-07-05      74         14  23.675281  3.907180  405.705811   
           2016-07-06     159         82  68.286506  5.863103  471.431509   
           2016-07-07     135         77  55.186046  5.440297  292.881704   
           2016-07-08     101         32  33.948965  4.651528  316.531584   
           2016-07-09      91         27  35.619995  4.480372  346.921538   
           2016-07-10      76         12  23.482151  3.921795  421.086579   
           2016-07-11     128         48  41.998025  5.164157  357.674766   
           2016-07-12      81         17  28.534357  4.234226  364.739383   
           2016-07-13      90         17  28.793148  4.328443  481.238778   
           2016-07-14     114         57  47.124817  4.948326  318.303947   
           2016-07-15      81         26  31.351232  4.225571  373.940741   
           2016-07-16     104         33  39.557444  4.796312  396.508462   
           2016-07-17      75         19  26.125288  4.060796  411.013733   
           2016-07-18     113         32  38.649965  4.951425  457.459381   
           2016-07-19      65         18  19.729651  3.611885  319.810154   
           2016-07-20     109         46  44.712683  4.834723  221.962936   

                         oversell_count  empty_rooms_simulated  \
hotel      arrival_date                                          
City Hotel 2016-07-01                42                     26   
           2016-07-02                17                     16   
           2016-07-03                19                      2   
           2016-07-04                22                      8   
           2016-07-05                14                      0   
           2016-07-06                50                     32   
           2016-07-07                40                     37   
           2016-07-08                26                      6   
           2016-07-09                21                      6   
           2016-07-10                15                      0   
           2016-07-11                37                     11   
           2016-07-12                17                      0   
           2016-07-13                21                      0   
           2016-07-14                31                     26   
           2016-07-15                17                      9   
           2016-07-16                27                      6   
           2016-07-17                15                      4   
           2016-07-18                31                      1   
           2016-07-19                11                      7   
           2016-07-20                29                     17   

                         walks_simulated  
hotel      arrival_date                   
City Hotel 2016-07-01                  0  
           2016-07-02                  0  
           2016-07-03                  0  
           2016-07-04                  0  
           2016-07-05                  0  
           2016-07-06                  0  
           2016-07-07                  0  
           2016-07-08                  0  
           2016-07-09                  0  
           2016-07-10                  3  
           2016-07-11                  0  
           2016-07-12                  0  
           2016-07-13                  4  
           2016-07-14                  0  
           2016-07-15                  0  
           2016-07-16                  0  
           2016-07-17                  0  
           2016-07-18                  0  
           2016-0

## 13. Save the policy-selection evidence

We save the overall policy table, hotel-level diagnostics, and (if a policy is selected) the nightly Choose decisions.

These are historical policy-selection records. They are not a future 14-night recommendation.

In [16]:
evidence_folder = Path("evidence/choose_policy_selection")
evidence_folder.mkdir(parents=True, exist_ok=True)

policy_scores.to_csv(
    evidence_folder / "choose_policy_scores.csv",
    index=False
)

hotel_policy_scores.to_csv(
    evidence_folder / "choose_hotel_policy_scores.csv",
    index=False
)

if selected_policy is not None:
    selected_nights.to_csv(
        evidence_folder / "choose_selected_policy_nights.csv"
    )

pd.DataFrame([{
    "selected_policy": (
        selected_policy["policy"] if selected_policy is not None else "NONE"
    ),
    "walk_cost_eur": contract.WALK_COST,
    "walk_ceiling_per_1000": contract.WALK_CEILING,
    "proof_used_for_selection": False
}]).to_csv(
    evidence_folder / "selection_record.csv",
    index=False
)

print("Saved policy-selection evidence to:", evidence_folder)

Saved policy-selection evidence to: evidence\choose_policy_selection


In [17]:
# Freeze everything Notebook 12 needs, so it reads this file instead of hand-copied values.
import datetime
import hashlib


def file_sha256(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def plain(row):
    return {k: (v.item() if hasattr(v, "item") else v) for k, v in row[comparison_columns].items()}


def describe(policy_name):
    for prefix, family, kind in [("model_buffer_", "model_buffer", "multiplier"), ("model_margin_", "model_margin", "margin"),
                                 ("flat_rule_margin_", "flat_rule_margin", "margin")]:
        if policy_name.startswith(prefix):
            value = float(policy_name[len(prefix):])
            return {"family": family, kind: int(value) if kind == "margin" else value}
    return {"family": policy_name}


# Declared Proof benchmark: the cheapest eligible model-based policy meeting the ceiling on Choose.
is_model = policy_scores["policy"].str.startswith("model_")
acceptable_model = policy_scores[is_model & policy_scores["ceiling_met"]]
best_model = acceptable_model.sort_values("cost_eur", kind="stable").iloc[0] if not acceptable_model.empty else None

frozen_configuration = {
    "frozen_at": datetime.datetime.now().isoformat(timespec="seconds"),
    "frozen_before_this_pipeline_scored_proof": True,
    "historical_disclosure": ("The Proof period was inspected in earlier project work (see docs/DECISIONS.md, 2026-11-18). "
                              "Notebook 12's evaluation is retrospective, not historically blind."),
    "model": BASE_MODEL_NAME,
    "model_settings": BASE_MODEL_SETTINGS,
    "features": feature_columns,
    "probability_method": PROBABILITY_METHOD,
    "calibrator_intercept": CAL_INTERCEPT if USE_CALIBRATED else None,
    "calibrator_coefficient": CAL_COEFFICIENT if USE_CALIBRATED else None,
    "selected_policy": selected_policy["policy"] if selected_policy is not None else None,
    "selected_policy_rule": describe(selected_policy["policy"]) if selected_policy is not None else None,
    "flat_rule_benchmark": ({"policy": best_flat["policy"], **describe(best_flat["policy"]),
                             "hotel_rates_from_train": {h: float(r) for h, r in hotel_rates.items()}}
                            if best_flat is not None else None),
    "model_policy_benchmark": ({"policy": best_model["policy"], **describe(best_model["policy"])} if best_model is not None else None),
    "rounding": "ceil, then subtract any margin, floored at zero",
    "selection_rule": "drop policies above the walk ceiling on Choose; take the lowest Choose cost",
    "choose_scores": {"selected": plain(selected_policy) if selected_policy is not None else None,
                      "flat_rule": plain(best_flat) if best_flat is not None else None},
    "inputs_sha256": {path: file_sha256(path) for path in ["evidence/hyperparameter_experiments/selected_settings.csv",
                                                           "evidence/probability_calibration/calibration_decision.json"]},
    "walk_cost_eur": contract.WALK_COST,
    "walk_ceiling_per_1000": contract.WALK_CEILING,
}
(evidence_folder / "frozen_configuration.json").write_text(json.dumps(frozen_configuration, indent=2, default=str) + chr(10), encoding="utf-8")
print("Frozen configuration written to", evidence_folder / "frozen_configuration.json")
print(json.dumps({k: frozen_configuration[k] for k in ["model", "model_settings", "probability_method", "selected_policy",
                                                       "selected_policy_rule", "flat_rule_benchmark", "model_policy_benchmark", "choose_scores"]}, indent=2, default=str))

Frozen configuration written to evidence\choose_policy_selection\frozen_configuration.json
{
  "model": "logistic_regression",
  "model_settings": {
    "C": 0.1
  },
  "probability_method": "calibrated",
  "selected_policy": "flat_rule_margin_17",
  "selected_policy_rule": {
    "family": "flat_rule_margin",
    "margin": 17
  },
  "flat_rule_benchmark": {
    "policy": "flat_rule_margin_17",
    "family": "flat_rule_margin",
    "margin": 17,
    "hotel_rates_from_train": {
      "City Hotel": 0.4166271384751605,
      "Resort Hotel": 0.2515113493783506
    }
  },
  "model_policy_benchmark": {
    "policy": "model_margin_18",
    "family": "model_margin",
    "margin": 18
  },
  "choose_scores": {
    "selected": {
      "policy": "flat_rule_margin_17",
      "cost_eur": 2414409.4,
      "empty_rooms": 5532,
      "walked_guests": 169,
      "walks_per_1000": 9.06,
      "ceiling_met": true
    },
    "flat_rule": {
      "policy": "flat_rule_margin_17",
      "cost_eur": 2414409.4,


## 14. Interpretation and decision record

Before accepting the selected policy, review:
- Does it meet the overall walk ceiling?
- Is the cost improvement meaningful compared with the simple historical rules?
- Does either hotel have a much higher walk rate?
- Is the model's probability estimate trustworthy enough to justify the added complexity?
- Did Notebook 09 show that calibration should be used?
- Does Notebook 10 suggest that the chosen uncertainty buffer is too narrow or too wide?
- Are the feature timing assumptions defensible?

Record the selected policy, the reason, and any caveats in `docs/DECISIONS.md`.

### Limitations

- The result is a historical simulation using the contract, not a physical capacity model.
- Booking rows are treated as room-equivalents by the supplied contract.
- Actual room inventory, rooms-per-booking, capacity constraints and actual walk records are absent.
- The model-based policies use the probability method adopted by Notebook 09's pre-declared rule (see `frozen_configuration.json`); the uncalibrated versions are reported for reference only.
- Choose has been used to select the policy, so it is not a final unbiased estimate of performance.
- Proof must be used only after the model and policy are frozen.

### Questions

1. Why is the walk ceiling a constraint rather than just another cost?
2. Why can a lower-cost policy be unacceptable?
3. What is the difference between expected cancellations and an oversell count?
4. Why does adding an uncertainty buffer increase both protection against empty rooms and the risk of walks?
5. Why should we inspect each hotel separately even when the contract uses a group-level guardrail?

**Next notebook:** `12_proof_evaluation.ipynb`, where we freeze the chosen model and policy and evaluate them on the reserved Proof period exactly once.